In [2]:
import dash_leaflet as dl
from dash import dcc, html, dash_table
from dash.dependencies import Input, Output, State
from jupyter_dash import JupyterDash
import plotly.express as px
import base64
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from CRUD_Python_Module import AnimalShelter

JupyterDash.infer_jupyter_proxy_config()

###########################
# Data Manipulation / Model
###########################

username = "aacuser"
password = "MyPassword123"

db = AnimalShelter(username, password)

df = pd.DataFrame.from_records(db.read({}))

if '_id' in df.columns:
    df.drop(columns=['_id'], inplace=True)

#########################
# Dashboard Layout / View
#########################

app = JupyterDash(__name__)

image_filename = 'Grazioso Salvare Logo.png'   
encoded_image = base64.b64encode(open(image_filename, 'rb').read()).decode()

app.layout = html.Div([
    html.Center(html.B(html.H1('CS-340 Dashboard'))),
    html.Center(html.H3('Chris')),

    html.Center(
        html.Img(
            src='data:image/png;base64,{}'.format(encoded_image),
            style={'width': '300px'}
        )
    ),

    html.Hr(),
    
    html.Div([
    dcc.RadioItems(
        id='filter-type',
        options=[
            {'label': 'All', 'value': 'ALL'},
            {'label': 'Water Rescue', 'value': 'WATER'},
            {'label': 'Mountain/Wilderness Rescue', 'value': 'MOUNTAIN'},
            {'label': 'Disaster/Tracking', 'value': 'DISASTER'}
        ],
        value='ALL',
        labelStyle={'display': 'inline-block', 'margin-right': '20px'}
    )
]),

html.Br(),

    dash_table.DataTable(
        id='datatable-id',
        columns=[{"name": i, "id": i, "deletable": False, "selectable": True} for i in df.columns],
        data=df.to_dict('records'),
        page_size=10,
        sort_action='native',
        filter_action='native',
        row_selectable='single',
        selected_rows=[0],
        style_table={'overflowX': 'auto'},
        style_cell={
            'textAlign': 'left',
            'minWidth': '120px',
            'width': '120px',
            'maxWidth': '180px',
            'whiteSpace': 'normal'
        }
    ),

    html.Br(),
    html.Hr(),

    html.Div(className='row',
             style={'display': 'flex'},
             children=[
                 html.Div(id='graph-id', className='col s12 m6'),
                 html.Div(id='map-id', className='col s12 m6')
             ])
])

@app.callback(
    Output('datatable-id', 'data'),
    [Input('filter-type', 'value')]
)
def update_dashboard(filter_type):

    if filter_type == 'WATER':
        query = {
            "animal_type": "Dog",
            "breed": {"$in": ["Labrador Retriever Mix", "Chesapeake Bay Retriever", "Newfoundland"]},
            "sex_upon_outcome": "Intact Female",
            "age_upon_outcome_in_weeks": {"$gte": 26, "$lte": 156}
        }

    elif filter_type == 'MOUNTAIN':
        query = {
            "animal_type": "Dog",
            "breed": {"$in": ["German Shepherd", "Alaskan Malamute", "Old English Sheepdog", "Siberian Husky", "Rottweiler"]},
            "sex_upon_outcome": "Intact Male",
            "age_upon_outcome_in_weeks": {"$gte": 26, "$lte": 156}
        }

    elif filter_type == 'DISASTER':
        query = {
            "animal_type": "Dog",
            "breed": {"$in": ["Doberman Pinscher", "German Shepherd", "Golden Retriever", "Bloodhound", "Rottweiler"]},
            "sex_upon_outcome": "Intact Male",
            "age_upon_outcome_in_weeks": {"$gte": 20, "$lte": 300}
        }

    else:  # ALL
        query = {}

    data = db.read(query)
    return data

@app.callback(
    Output('datatable-id', 'style_data_conditional'),
    [Input('datatable-id', 'selected_columns')]
)
def update_styles(selected_columns):
    if selected_columns is None:
        return []

    return [{
        'if': {'column_id': i},
        'background_color': '#D2F3FF'
    } for i in selected_columns]

@app.callback(
    Output('map-id', "children"),
    [Input('datatable-id', "derived_virtual_data"),
     Input('datatable-id', "derived_virtual_selected_rows")]
)
def update_map(viewData, index):
    if viewData is None or len(viewData) == 0:
        return []

    dff = pd.DataFrame.from_dict(viewData)

    if index is None or len(index) == 0:
        row = 0
    else:
        row = index[0]

    return [
        dl.Map(style={'width': '1000px', 'height': '500px'},
               center=[30.75, -97.48],
               zoom=10,
               children=[
                   dl.TileLayer(id="base-layer-id"),
                   dl.Marker(
                       position=[dff.iloc[row, 13], dff.iloc[row, 14]],
                       children=[
                           dl.Tooltip(dff.iloc[row, 4]),
                           dl.Popup([
                               html.H1("Animal Name"),
                               html.P(dff.iloc[row, 9])
                           ])
                       ]
                   )
               ])
    ]
@app.callback(
    Output('graph-id', "children"),
    [Input('datatable-id', "derived_virtual_data")]
)
def update_graphs(viewData):

    if viewData is None or len(viewData) == 0:
        return []

    dff = pd.DataFrame(viewData)

    # Count top breeds only
    top_breeds = dff['breed'].value_counts().nlargest(10)

    return [
        dcc.Graph(
            figure=px.pie(
                names=top_breeds.index,
                values=top_breeds.values,
                title='Top 10 Dog Breeds'
            )
        )
    ]

app.run_server()

Dash app running on https://nylongrace-salutepenguin-3000.codio.io/proxy/8050/
